# 05 Predict with M9

Scores every complete site-day with the frozen M9 revision 2 scorer (label-free, imported from `m9_dev/`), then for each fold fits the two calibration coefficients on the other stations of the same cohort and decides the held-out station with the public control c = 0.7.

Abbreviations used here: **RPF** is reverse power flow, the condition where a distribution substation exports power because rooftop solar exceeds local demand; a *wrong RPF sign* is a meter recording that stores the export as an import. **M7** is the deterministic threshold rule, **M8** the two-stage XGBoost classifier and **M9** the compact counterfactual method (revision 2). **MW** and **MWh** are megawatts and megawatt-hours; one interval is 15 minutes.

**Inputs.** The fold manifest and the two frozen datasets; the scorer `m9_dev/m9_scorer.py`.

**Outputs.** `outputs/01_final_evaluation/05_m9/`: `scores_<cohort>.parquet` (cached label-free scores), `site_days_m9.parquet` (probability, outcome, window per site-day), `calibration_fits.csv` (`cal_intercept`, `cal_slope` and the raw thresholds per fold), `intervals_m9.parquet`; `manifests/05_m9_predict.json`.

**Approximate runtime.** About one minute.

**Prerequisites.** Notebook 01.

**Main process.**

1. Compute the label-free sigma floor per cohort and score every site-day once.
2. Per fold, fit the calibration on headline-confidence days of the calibration stations; apply to the held-out station.
3. Expand the decisions to the common interval schema.

## 1. Setup

In [ ]:
import sys
from pathlib import Path

import pandas as pd
from IPython.display import Image, Markdown, display


def article_root() -> Path:
    """Locate publication/2_journal_article from JupyterLab, VS Code or the repository root."""
    for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        if (candidate / "final_eval" / "cli.py").exists():
            return candidate
        nested = candidate / "publication" / "2_journal_article"
        if (nested / "final_eval" / "cli.py").exists():
            return nested
    raise FileNotFoundError("Could not locate publication/2_journal_article.")


ARTICLE = article_root()
sys.path.insert(0, str(ARTICLE))
sys.path.insert(0, str(ARTICLE.parents[1] / "src"))  # the repository's pynrpf package

from final_eval import cli, config  # noqa: E402

SETTINGS = config.load()  # verifies the frozen dataset hashes
OUT = SETTINGS.output_root()
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
print("Article root:", ARTICLE.relative_to(ARTICLE.parents[2]))

## 2. Score and calibrate

The calibration is the only fitted part of M9: p = 1 / (1 + exp(−(cal_intercept + cal_slope · z))) with z the signed-log evidence. Beta `unsure` days are scored and decided but never enter a fit.

In [ ]:
table = cli.stage_predict(SETTINGS, "m9")
fits = pd.read_csv(OUT / "05_m9" / "calibration_fits.csv")
display(fits.round(4))

## 3. Agreement with the frozen development run

The same folds, code and data were used in `m9_dev/runs/phase5_final_rev2`; the calibration coefficients must match to four decimals. This is the check that the port changed nothing.

In [ ]:
frozen = pd.read_csv(ARTICLE / "m9_dev" / "runs" / "phase5_final_rev2" / "calibration_fits.csv")
merged = fits.merge(frozen, on=["cohort", "held_out"], suffixes=("", "_frozen"))
assert (merged["cal_intercept"] - merged["alpha"]).abs().max() < 1e-4
assert (merged["cal_slope"] - merged["beta"]).abs().max() < 1e-4
print("Calibration fits match the frozen run.")

## Conclusion

M9 decisions are written for every held-out site-day. Notebook 06 joins them with M7 and M8.